# LCLDD: Lyapunov-based Constrained Latent Distillation
## Compressing Deep Reasoning into Continuous Latent Trajectories
---
*Meeting Presentation Notes*


### 1. Motivation & The Core Problem
Large reasoning models (like OpenAI's o1 or DeepSeek's R1) arrive at incredible conclusions by generating massive strings of "Chain of Thought" (CoT) text tokens.
However:
* CoT text generation is **computationally expensive** (O(N^2) attention over hundreds of intermediate reasoning tokens).
* We want smaller, faster models that can mimic this reasoning logic *without* the verbose text overhead.

**Our Goal:** Distill the deep reasoning capabilities of a large Teacher model into a much smaller Student model (e.g., 1.5B parameters) by training it to reason entirely inside continuous vector spaces (Latent Space) before decoding a single output word.


### 2. Our Novel Architecture
Instead of teaching the Student to output text explanations, we train it to loop a continuous state vector $h_t$.

* **Step A: Precomputing Teacher States**
  We run a 3B parameter Teacher model over difficult reasoning problems (like GSM8K). We capture and cache the intermediate hidden states as "golden trajectories".
  
* **Step B: Student Dynamic Thinking**
  The Student Model (Qwen2.5-1.5B) is embedded with a custom `ThinkingBlock`. Given a problem, it generates an initial thought vector $\phi(x)$. It iteratively updates this vector for $T_{max}$ continuous steps.

* **Step C: Latent Injection**
  Once the "thought" is complete, a `ProjectionHead` scales this vector and **injects it directly into the raw embedding manifold** of the last prompt token. The Student's standard LLM trunk then generates the final answer in one shot based on this enriched embedding!


### 3. The Mathematics of Stability: Lyapunov Energy
Because our Student is recursively updating a vector for multiple steps inside a loop, it is functionally a Nonlinear Dynamical System. Unconstrained, these vectors easily explode (Gradient Explosion) or diverge.

**Our Solution:** 
We heavily drew upon Control Theory to formulate a custom **Lyapunov Function**. During training, we impose a steep penalty if the "Energy" of the thinking trajectory does not monotonically descend. 
By forcing $\Delta E < 0$, we mathematically guarantee that the latent "thought" trajectory converges to a stable attractor state, preventing hallucinations and stabilizing inference!


### 4. The "Winning Configuration"
After extensive empirical iteration, we arrived at a highly optimized training strategy:
1. **Raw Embedding Extraction:** We abandoned extracting high-level hidden states. We now extract $\phi(x)$ from the absolute raw layer embeddings of the absolute last prompt token. `phi_x = student.get_input_embeddings()(...)[:, -1, :]`
2. **Precision Injection Geometry:** We inject the final reasoned state delta *exclusively* onto the last embedding token, leaving the rest of the prompt context unaffected.
3. **Scaling Factor:** The thinking delta is bounded via Tanh and forcefully scaled to exactly `10% (0.1)` of the standard embedding layer magnitude to prevent coordinate disruption (`injection_scale = 0.1`).


### 5. Quick Look: Training Logic (train.py excerpt)
Below is the core of how the logic translates to PyTorch code during our End-to-End training loop:


In [ ]:
# 1. Project the final "Thought" into embedding space dimensions
delta = proj_head(h_T, phi_x_i).to(student.dtype)

# 2. Scale our thought vector to precisely 10% of standard embedding norm lengths
embed_norm = orig.norm(dim=-1).mean()
delta_norm = delta.norm(dim=-1).mean()
delta = delta * (embed_norm / (delta_norm + 1e-8)) * config["injection_scale"]

# 3. Targeted injection into LAST token of the origin prompt
modified_embeds = orig.clone()
modified_embeds[:, -1:, :] = orig[:, -1:, :] + delta.unsqueeze(1)

# 4. Standard Causal generation using enriched embeddings
outputs = student(
    inputs_embeds=modified_embeds,
    attention_mask=extended_mask,
    output_hidden_states=False
)



### 6. Summary & Next Steps
* **Results so far:** We successfully built the framework, isolated the Lyapunov logic, and verified the gradient flow through the injected reasoning tokens.
* **Next target:** Scaling up the training set size to measure exact zero-shot improvement bounds on benchmarks vs baseline.

